In [4]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
import os


drive_path = '/content/drive/MyDrive/udemy-gerçek projeler/drd'
for root, dirs, files in os.walk(drive_path):
    for file in files:
        if file.endswith('.zip'):
            print(os.path.join(root, file))

In [6]:
import zipfile

zip_path = '/content/drive/MyDrive/udemy-gerçek projeler/drd.zip'  # ZIP pathway
extract_path = '/content/dataset'  # output directory

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

In [8]:
import os
import shutil
import pandas as pd

#Paths
csv_path = "/content/dataset/drd/image_data.csv"
source_folder = "/content/dataset/drd/processed_images"
target_root = "/content/dataset/drd/processed_dataset"


df = pd.read_csv(csv_path)


for index, row in df.iterrows():
    filename = row['filename']
    label = str(row['label'])
    set_type = row['set']           # "train" or "val"

    src = os.path.join(source_folder, filename)
    dest_dir = os.path.join(target_root, set_type, label)
    dest = os.path.join(dest_dir, filename)

    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy2(src, dest)
    else:
        print(f"ERROR: {filename} was not found, skipped.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/dataset/drd/image_data.csv'

In [9]:

import os
import shutil
import pandas as pd

# paths
csv_path = "/content/dataset/drd/train.csv"
source_folder = "/content/dataset/drd/train_images"
target_root = "/content/dataset/organized"


df = pd.read_csv(csv_path)

# === If columns do not exist, create them ===
if 'filename' not in df.columns:
    df['filename'] = df['id_code'] + '.png'

if 'label' not in df.columns:
    df['label'] = df['diagnosis'].astype(str)

if 'set' not in df.columns:

    df['set'] = df['label'].apply(lambda x: 'train' if int(x) < 4 else 'val')

# === Move files to appropriate folders ===
for index, row in df.iterrows():
    filename = row['filename']
    label = str(row['label'])
    set_type = row['set']  # "train", "val"

    src = os.path.join(source_folder, filename)
    dest_dir = os.path.join(target_root, set_type, label)
    dest = os.path.join(dest_dir, filename)

    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy2(src, dest)
    else:
        print(f"Directory was not found: {src}")


In [10]:
import os
import shutil
import pandas as pd

# === PATHS ===
csv_path = "/content/dataset/drd/train.csv"
source_folder = "/content/dataset/drd/train_images"
target_root = "/content/dataset/organized"


df = pd.read_csv(csv_path)


if 'filename' not in df.columns and 'id_code' in df.columns:
    df['filename'] = df['id_code'] + '.png'

if 'label' not in df.columns and 'diagnosis' in df.columns:
    df['label'] = df['diagnosis'].astype(str)

if 'set' not in df.columns:
    df['set'] = 'train'

for index, row in df.iterrows():
    filename = row['filename']
    label = str(row['label'])
    set_type = row['set']  # "train", "val", "test"

    src = os.path.join(source_folder, filename)
    dest_dir = os.path.join(target_root, set_type, label)
    dest = os.path.join(dest_dir, filename)

    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy2(src, dest)
    else:
        print(f"❗File was not found: {src}")



In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Original CSV
df = pd.read_csv("/content/dataset/drd/train.csv")

df['filename'] = df['id_code'] + '.jpg'
df['label'] = df['diagnosis'].astype(str)

df = df[['filename', 'label']]

# %80 train, %20 val
train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)


train_df['set'] = 'train'
val_df['set'] = 'val'


final_df = pd.concat([train_df, val_df], ignore_index=True)

final_df.to_csv("/content/dataset/image_data.csv", index=False)
print("CSV file is created: image_data.csv")

CSV dosyası oluşturuldu: image_data.csv


In [12]:
import os
import shutil
import pandas as pd

# === PATHS ===
csv_path = "/content/dataset/image_data.csv"
source_folder = "/content/dataset/drd/processed_images"
target_root = "/content/dataset/drd/processed_dataset"


df = pd.read_csv(csv_path)

ü
for index, row in df.iterrows():
    filename = row['filename']
    label = str(row['label'])
    set_type = row['set']           # "train" ya da "val"

    src = os.path.join(source_folder, filename)
    dest_dir = os.path.join(target_root, set_type, label)
    dest = os.path.join(dest_dir, filename)

    os.makedirs(dest_dir, exist_ok=True)

    if os.path.exists(src):
        shutil.copy2(src, dest)
    else:
        print(f"HATA: {filename} bulunamadı, atlandı.")


HATA: 82910bba4753.jpg bulunamadı, atlandı.


In [14]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

def get_train_datagen():
    return ImageDataGenerator(
        rescale=1.0 / 255,


        horizontal_flip=True,

        vertical_flip=True,


    )
def get_test_val_datagen():
    return ImageDataGenerator(rescale=1. / 255)

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB5
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_dir = "/content/dataset/drd/processed_dataset/train"
val_dir = "/content/dataset/drd/processed_dataset/val"


IMAGE_SIZE = (456, 456)  # convenient for EfficientNetB5
BATCH_SIZE = 16
NUM_CLASSES = 5
EPOCHS = 30



model_save_path = '/content/dataset/efficientnetb5_model.h5'


train_datagen = get_train_datagen()
val_datagen = get_test_val_datagen()

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical'
)

base_model = EfficientNetB5(weights='imagenet', include_top=False, input_shape=(456, 456, 3))

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
predictions = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=predictions)


model.compile(optimizer=Adam(learning_rate=1e-4),
              loss='categorical_crossentropy',
              metrics=['accuracy'])


callbacks = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    ModelCheckpoint(model_save_path, save_best_only=True)
]


model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

print("Model training completed and saved:", model_save_path)


Found 2929 images belonging to 5 classes.
Found 732 images belonging to 5 classes.


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
124/184 ━━━━━━━━━━━━━━━━━━━━ 58s 978ms/step - accuracy: 0.5915 - loss: 1.0973

In [1]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']

final_train_acc = acc[-1]
final_val_acc = val_acc[-1]


with open("/content/model_accuracy.txt", "w") as f:
    f.write(f"Final Train Accuracy: {final_train_acc:.4f}\n")
    f.write(f"Final Validation Accuracy: {final_val_acc:.4f}\n")


GPU kullanılıyor mu? []


In [18]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# === Girdi ve çıktı klasörleri ===
input_path = "/content/dataset/drd/test_images"
output_path = "/content/dataset/drd/processed_testimages"

# ⛏️ Klasörü oluştur
os.makedirs(output_path, exist_ok=True)

# === Dosya listesi
files = os.listdir(input_path)
img_list = []

for filename in tqdm(files):
    img_path = os.path.join(input_path, filename)
    image = cv2.imread(img_path)

    if image is None:
        print(f"{filename} okunamadı, atlandı.")
        continue

    image = cv2.resize(image, (400, 400))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    img_list.append(image)

    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    thresh = cv2.threshold(blur, 10, 255, cv2.THRESH_BINARY)[1]

    contours, _ = cv2.findContours(thresh.copy(), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        print(f"{filename} → Kontur bulunamadı.")
        continue

    contour = max(contours, key=cv2.contourArea)
    if contour.shape[0] < 5:
        print(f"{filename} → Kontur çok küçük.")
        continue

    contour = contour[:, 0, :]
    x1, x2 = contour[:, 0].min(), contour[:, 0].max()
    y1, y2 = contour[:, 1].min(), contour[:, 1].max()

    w = x2 - x1
    h = y2 - y1

    if w < 10 or h < 10:
        print(f"{filename} → Kırpma alanı çok küçük.")
        continue

    x_margin = int(w * 0.04)
    y_margin = int(h * 0.05)
    x_start = x1 + x_margin
    x_end = x2 - x_margin
    y_start = y1 + y_margin
    y_end = y2 - y_margin

    if x_end <= x_start or y_end <= y_start:
        print(f"{filename} → Geçersiz kırpma koordinatları.")
        continue

    img_crop = image[y_start:y_end, x_start:x_end]
    if img_crop.size == 0:
        print(f"{filename} → img_crop boş.")
        continue

    img_crop = cv2.resize(img_crop, (400, 400))

    lab = cv2.cvtColor(img_crop, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=7.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    limg = cv2.merge((cl, a, b))
    final = cv2.cvtColor(limg, cv2.COLOR_LAB2RGB)

    med = cv2.medianBlur(final, 3)
    med_backg = cv2.medianBlur(final, 37)
    mask = cv2.addWeighted(med, 1, med_backg, -1, 255)
    result = cv2.bitwise_and(mask, med)

    # RGB'den BGR'ye çevirip .jpg olarak kaydet
    result_bgr = cv2.cvtColor(result, cv2.COLOR_RGB2BGR)
    save_path = os.path.join(output_path, filename.replace('.png', '.jpg'))

    cv2.imwrite(save_path, result_bgr)



100%|██████████| 1928/1928 [01:47<00:00, 17.93it/s]
